In [27]:
#Create an agent that classifies reviews as positive or negative using a simple rule-based approach.
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [28]:
llm = ChatOllama(
    model = "qwen3",
    base_url="http://localhost:11434"
)
prompt = ChatPromptTemplate.from_template(""" 
Given the user review below, classify it as either positive or negative based on the sentiment expressed in the review. Return only the classification in one word max without any additional explanation.
User Review: {review}
Classification (positive/negative):
""")
review_chain  = prompt | llm | StrOutputParser()    
review = "I absolutely loved this product! It exceeded my expectations and I would highly recommend it to anyone."
#review = "Unfortunately, this product did not meet my expectations. It was poorly made and I would not recommend it to anyone."
classification = review_chain.invoke({"review": review})
print(f"Review: {review}")
print(f"Classification: {classification}")

Review: I absolutely loved this product! It exceeded my expectations and I would highly recommend it to anyone.
Classification: positive


In [30]:
positive_review = "I absolutely loved this product! It exceeded my expectations and I would highly recommend it to anyone."

positive_reply_prompt = ChatPromptTemplate.from_template(""" You are an expert in writing positive review replies.
Given the user review , generate a positive reply to the user.You should encourage the user to share their experience on social media and thank them for their feedback.
User Review: {positive_review}
Positive Reply:
""")
positive_reply_chain = positive_reply_prompt | llm | StrOutputParser()
positive_reply = positive_reply_chain.invoke({"positive_review": positive_review})
print(f"Positive Reply: {positive_reply}")

Positive Reply: Thank you so much for your wonderful feedback! We’re thrilled to hear that the product exceeded your expectations—it means the world to our team to know we’ve made such a positive impact. 🌟 We’d love for you to share your experience on social media (tag us with #ProductName for a chance to be featured!) and spread the joy! Your support truly drives us to keep creating amazing things. If you ever need anything else, we’re here for you. Happy shopping! 😊


In [31]:
negative_review = "Unfortunately, this product did not meet my expectations. It was poorly made and I would not recommend it to anyone."
negative_reply_prompt = ChatPromptTemplate.from_template("""
You are an expert in replying to negative reviews in a calm and professional manner.
Given the user review , generate a calm  and professional reply to the user.
You should apologize for the inconvenience and ask them to send an email to support@agent.com for further assistance.
User Review: {negative_review}
Reply:
""")
negative_reply_chain = negative_reply_prompt | llm | StrOutputParser()
calm_reply = negative_reply_chain.invoke({"negative_review": negative_review})
print(f"Calm Reply: {calm_reply}")

Calm Reply: Thank you for sharing your feedback. We sincerely apologize for the inconvenience caused and regret that the product did not meet your expectations. We take customer satisfaction seriously and appreciate your input as we strive to improve. Please reach out to our support team at support@agent.com so we can assist you further and resolve this matter to your satisfaction. Your feedback is valuable, and we hope to make things right.


In [32]:
def route(info):
    if "positive" in info["sentiment"].lower():
        print("Routing to positive reply chain")
        return positive_reply_chain          
    else:
        print("Routing to negative reply chain")
        return negative_reply_chain

In [33]:
route({"sentiment": "positive"})

Routing to positive reply chain


ChatPromptTemplate(input_variables=['positive_review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['positive_review'], input_types={}, partial_variables={}, template=' You are an expert in writing positive review replies.\nGiven the user review , generate a positive reply to the user.You should encourage the user to share their experience on social media and thank them for their feedback.\nUser Review: {positive_review}\nPositive Reply:\n'), additional_kwargs={})])
| ChatOllama(model='qwen3', base_url='http://localhost:11434')
| StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableLambda
actual_review = "I absolutely loved this product! It exceeded my expectations and I would highly recommend it to anyone."
#actual_review = "Unfortunately, this product did not meet my expectations. It was poorly made and I would not recommend it to anyone."

#I have used actual_review and user_review to understand how to pass different inputs to different chains based on the 
#initial review that gets passed in the first invoke call in full_chain
#In this case the review_chain is expecting a variable called review
#And the positive_reply_chain and negative_reply_chain are expecting variables called positive_review and negative_review respectively.
#I can avoid all these by simply using review as the variable name across all chains and passing the same variable in the invoke call. But I wanted to demonstrate how to pass different variables to different chains in the full_chain.

full_chain = {
    "sentiment":RunnableLambda(lambda x: {"review": x['actual_review']}) | review_chain, 
    "positive_review": lambda x: x['user_review'],
    "negative_review": lambda x: x['user_review']
    } | RunnableLambda(route) 

output  = full_chain.invoke({"actual_review": actual_review, "user_review": actual_review})
print(output)

Routing to positive reply chain
Thank you so much for your kind words! We’re thrilled to hear your product exceeded your expectations—it means the world to us that you’re loving it. 🌟 Your support is invaluable, and we’d love for you to share your experience with your friends and followers on social media! Don’t forget to tag us and use the hashtag #ProductNameMagic for a chance to be featured! 😊 Thanks again for your feedback, and feel free to reach out if you need anything else—we’re here to help!
